In [ ]:
# ============================================================
# 00_config_and_manifest
# ============================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import subprocess
import platform
import random
from datetime import datetime, timezone

import torch
import numpy as np
import pandas as pd

# -- Personal Libraries (idéntico stack que hparam-search.ipynb / nn_final_evaluation.ipynb)
from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory, SplineBuilder
from src.nn.spline import build_price_basis
from src.nn.models import IntegrableDemandHead, ICDN
from src.nn.loss import ElasticityLoss
from src.multiproduct import MultiProductDataset, ProductTokenBuilder
from src.utils import TemporalSplitter, BlockBootstrapSampler

# ── SMOKE TEST TOGGLE ────────────────────────────────────────────────
# True  -> pipeline reducido (pocos folds/seeds/configs) para validar
#          que las 12 secciones corren de punta a punta sin errores.
# False -> escala completa descrita en el diseño experimental.
SMOKE_TEST = True

BASE_SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}  |  SMOKE_TEST={SMOKE_TEST}")

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(BASE_SEED)

# ── Data / model dims (idénticos a hparam-search.ipynb, no son objeto de estudio) ──
N_UPCS        = 5
SMOOTH_WINDOW = 8
BETA_EDA      = -2
K_NEIGHBORS   = 5
D_STORE, D_BRAND, D_STYLE = 16, 8, 8

# ── Matriz principal de arquitecturas (Propuestas 1 + 2) ─────────────
MODEL_MATRIX = [
    {"model_id": "TP-DOT", "basis_type": "truncated_cubic", "score_mode": "scaled_dot"},
    {"model_id": "NC-DOT", "basis_type": "natural_cubic",   "score_mode": "scaled_dot"},
    {"model_id": "TP-ADD", "basis_type": "truncated_cubic", "score_mode": "additive"},
    {"model_id": "NC-ADD", "basis_type": "natural_cubic",   "score_mode": "additive"},  # factorial diagnóstico
]
CONFIRMATORY_MODELS = ["TP-DOT", "NC-DOT", "TP-ADD"]        # matriz principal (3)
SCREENING_MODELS    = ["TP-DOT", "NC-DOT", "TP-ADD"]        # equal-budget retuning (3)
# NC-ADD sólo se ejecuta en fixed-config substitution (diagnóstico factorial),
# no en el screening/confirmatory de coste completo — evita duplicar presupuesto.

# ── Folds y seeds ─────────────────────────────────────────────────────
if SMOKE_TEST:
    N_INNER_FOLDS      = 1     # normal: 3
    N_OUTER_FOLDS      = 2     # normal: 5
    SCREENING_SEEDS    = [11]  # normal: 1 seed, igual
    CONFIRMATORY_SEEDS = [11, 29]           # normal: [11,29,42,77,123]
    N_COMMON_CONFIGS   = 2     # normal: 8
else:
    N_INNER_FOLDS      = 3
    N_OUTER_FOLDS      = 5
    SCREENING_SEEDS    = [11]
    CONFIRMATORY_SEEDS = [11, 29, 42, 77, 123]
    N_COMMON_CONFIGS   = 8

MIN_TRAIN_FRAC   = 0.50
SAMPLER_SEED     = 2026  # semilla única para muestrear el grid común (sección 03)

# ── Training budget (idéntico a hparam-search.ipynb; NO varía entre arquitecturas) ──
N_EPOCHS_P0 = 250
N_EPOCHS_P1 = 300
PATIENCE    = 20
ES_PATIENCE = 40

# ── Paths ──────────────────────────────────────────────────────────────
RESULTS_DIR = Path("../results/architecture_alternatives")
CKPT_DIR    = RESULTS_DIR / "checkpoints"
for p in [RESULTS_DIR, CKPT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH        = RESULTS_DIR / "run_manifest.json"
FIXED_CONFIG_PATH     = RESULTS_DIR / "fixed_config_runs.parquet"
SEARCH_RUNS_PATH      = RESULTS_DIR / "search_runs.parquet"
OUTER_METRICS_PATH    = RESULTS_DIR / "outer_metrics.parquet"
ELASTICITIES_PATH     = RESULTS_DIR / "elasticities.parquet"
ATTENTION_EDGES_PATH  = RESULTS_DIR / "attention_edges.parquet"
SUMMARY_TABLE_PATH    = RESULTS_DIR / "summary_table.csv"

def _git_commit_hash() -> str | None:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=Path.cwd().parent, text=True
        ).strip()
    except Exception:
        return None

# El manifest se construye incrementalmente: cada sección posterior añade
# su propia clave (split_plan, hparam_grid, run counts, timings...) y se
# reescribe con save_manifest(). Así el fichero final documenta el
# experimento completo, no sólo la config inicial.
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "smoke_test": SMOKE_TEST,
    "git_commit": _git_commit_hash(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": device,
    "base_seed": BASE_SEED,
    "model_matrix": MODEL_MATRIX,
    "confirmatory_models": CONFIRMATORY_MODELS,
    "screening_models": SCREENING_MODELS,
    "budget": {
        "n_inner_folds": N_INNER_FOLDS,
        "n_outer_folds": N_OUTER_FOLDS,
        "screening_seeds": SCREENING_SEEDS,
        "confirmatory_seeds": CONFIRMATORY_SEEDS,
        "n_common_configs": N_COMMON_CONFIGS,
        "min_train_frac": MIN_TRAIN_FRAC,
        "sampler_seed": SAMPLER_SEED,
        "n_epochs_p0": N_EPOCHS_P0,
        "n_epochs_p1": N_EPOCHS_P1,
        "patience": PATIENCE,
        "es_patience": ES_PATIENCE,
    },
}

def save_manifest():
    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2, default=str)

save_manifest()
print(f"Manifest written to {MANIFEST_PATH}")

In [ ]:
# ============================================================
# 01_load_fixed_split_plan
# ============================================================

# ── Loader (idéntico a hparam-search.ipynb) ───────────────────────────
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv").copy()
print(f"Dataset shape: {df.shape}")

encoder = ColumnEncoder()
_, store_cats = encoder.factorize(df, "store_code", sort=True)
_, week_cats  = encoder.factorize(df, "week_id", sort=True)
_, brand_cats = encoder.factorize(df, "brand_family_norm",  sort=True)
_, style_cats = encoder.factorize(df, "style_segment_norm", sort=True)

n_stores, n_weeks, n_brands, n_styles = len(store_cats), len(week_cats), len(brand_cats), len(style_cats)
print(f"Stores: {n_stores}  |  Weeks: {n_weeks}  |  Brands: {n_brands}  |  Styles: {n_styles}")

brand_map = {v: i + 1 for i, v in enumerate(brand_cats)}
style_map = {v: i + 1 for i, v in enumerate(style_cats)}
df["brand_family_norm"]  = df["brand_family_norm"].map(brand_map).fillna(0).astype(int)
df["style_segment_norm"] = df["style_segment_norm"].map(style_map).fillna(0).astype(int)

mp_builder = MultiProductBuilder()
mp_builder.fit(df, n_upcs=N_UPCS)
full_wide_raw = mp_builder.transform().copy()
n_upcs = mp_builder.n
print(f"Full wide shape: {full_wide_raw.shape}  |  UPCs: {mp_builder.selected_upcs[:N_UPCS]}")

store_map = {v: i for i, v in enumerate(store_cats)}
week_map  = {v: i for i, v in enumerate(week_cats)}

# ── Neighbor metadata (igual que hparam-search.ipynb) ─────────────────
upc_meta = (
    df.groupby("upc_code")[["category_code", "brand_family_norm",
                             "style_segment_norm", "liters_per_upc"]]
    .first()
    .loc[mp_builder.selected_upcs]
)
cat_codes, _ = pd.factorize(upc_meta["category_code"], sort=True)
neighbor_meta = {
    "category": torch.tensor(cat_codes, dtype=torch.long, device=device),
    "brand":    torch.tensor(upc_meta["brand_family_norm"].values, dtype=torch.long,    device=device),
    "style":    torch.tensor(upc_meta["style_segment_norm"].values, dtype=torch.long,    device=device),
    "liters":   torch.tensor(upc_meta["liters_per_upc"].values,     dtype=torch.float32, device=device),
}

# ── Fixed split plan ───────────────────────────────────────────────────
# outer_fold_splits:  N_OUTER_FOLDS expanding folds -> confirmatory (sección 06)
#                      y también sirve como fold único de referencia para
#                      fixed-config substitution (sección 04, usa fold 0).
# inner_fold_splits:  N_INNER_FOLDS expanding folds sobre el TRAIN del primer
#                      outer fold -> screening / equal-budget retuning (sección 05).
splitter = TemporalSplitter(week_col="week_id")

outer_fold_splits = splitter.expanding_splits(
    df=full_wide_raw, n_folds=N_OUTER_FOLDS, min_train_frac=MIN_TRAIN_FRAC,
)
# Inner folds del screening se generan SOLO a partir del train del outer fold 0,
# para que el screening nunca vea datos del rango temporal usado en confirmatory.
inner_fold_splits = splitter.expanding_splits(
    df=outer_fold_splits[0][0], n_folds=N_INNER_FOLDS, min_train_frac=MIN_TRAIN_FRAC,
)

def _fold_week_ids(fold_splits):
    return [
        {
            "train_weeks": sorted(train_fold["week_id"].unique().tolist()),
            "val_weeks":   sorted(val_fold["week_id"].unique().tolist()),
            "n_train": len(train_fold), "n_val": len(val_fold),
        }
        for train_fold, val_fold in fold_splits
    ]

manifest["split_plan"] = {
    "week_col": "week_id",
    "min_train_frac": MIN_TRAIN_FRAC,
    "outer_folds": _fold_week_ids(outer_fold_splits),
    "inner_folds": _fold_week_ids(inner_fold_splits),
}
save_manifest()

print(f"Outer folds (confirmatory): {len(outer_fold_splits)}")
for i, (tr, va) in enumerate(outer_fold_splits):
    print(f"  outer fold {i}: train={len(tr):,} val={len(va):,} "
          f"train_weeks={tr['week_id'].nunique()} val_weeks={va['week_id'].nunique()}")

print(f"Inner folds (screening, sobre train del outer fold 0): {len(inner_fold_splits)}")
for i, (tr, va) in enumerate(inner_fold_splits):
    print(f"  inner fold {i}: train={len(tr):,} val={len(va):,} "
          f"train_weeks={tr['week_id'].nunique()} val_weeks={va['week_id'].nunique()}")

In [ ]:
# ============================================================
# 02_validate_code_fixes
# ============================================================
def _build_smoke_model(basis_type: str, score_mode: str, n: int = N_UPCS) -> ICDN:
    """Builds a minimal ICDN instance for the given (basis_type, score_mode)
    combination, using synthetic log-price data just to derive valid knots."""
    rng = np.random.default_rng(0)
    builder = SplineBuilder()
    spline_configs = []
    for _ in range(n):
        x_i = rng.normal(loc=0.5, scale=0.2, size=500)
        spline_configs.append(
            builder.build_from_data(x_i, n_basis=6, q_min=0.05, q_max=0.95, basis_type=basis_type)
        )
    price_splines = build_price_basis(basis_type, spline_configs)

    token_builder = ProductTokenBuilder(
        n=n, n_stores=8, d_store=D_STORE,
        n_brands=4, d_brand=D_BRAND, n_styles=3, d_style=D_STYLE,
    )
    head = IntegrableDemandHead(
        context_dim=token_builder.d_token, K_splines=price_splines.K, n=n,
        k_neighbors=min(K_NEIGHBORS, n - 1), hidden=(32, 16), act="gelu", dropout=0.0,
        use_cross=True, enforce_negative_beta=True,
        attention_score_mode=score_mode, same_category_strict=False,
    )
    return ICDN(context_builder=token_builder, price_splines=price_splines, head=head, n=n).to(device)

def _smoke_batch(n: int, B: int = 8) -> dict:
    return {
        "store_code": torch.randint(0, 8, (B,), device=device),
        "week_id":    torch.randint(0, 20, (B,), device=device),
        "prices":     torch.randn(B, n, device=device) * 0.1 + 0.5,
        "demands":    torch.randn(B, n, device=device),
        "obs_mask":   torch.ones(B, n, device=device),
        **{f"promo_{i}": torch.zeros(B, dtype=torch.long, device=device) for i in range(n)},
    }

print("Validating all (basis_type, score_mode) combinations in MODEL_MATRIX...")
validation_report = []
for spec in MODEL_MATRIX:
    ok, err = True, None
    try:
        model = _build_smoke_model(spec["basis_type"], spec["score_mode"])
        batch = _smoke_batch(n=N_UPCS)
        # Forward BEFORE freeze_graph (dense path — this is exactly where
        # Bug 2 above crashed) and AFTER freeze_graph (sparse path).
        y_hat, eps_hat, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta={
            "category": torch.zeros(N_UPCS, dtype=torch.long, device=device),
            "brand":    torch.zeros(N_UPCS, dtype=torch.long, device=device),
            "style":    torch.zeros(N_UPCS, dtype=torch.long, device=device),
            "liters":   torch.ones(N_UPCS, dtype=torch.float32, device=device),
        })
        assert y_hat.shape == (8, N_UPCS), f"y_hat shape {y_hat.shape}"
        assert torch.isfinite(y_hat).all(), "y_hat has NaN/Inf"
        assert torch.isfinite(aux["E"]).all(), "E has NaN/Inf"
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    validation_report.append({"model_id": spec["model_id"], "ok": ok, "error": err})
    status = "OK" if ok else "FAILED"
    print(f"  [{status}] {spec['model_id']} (basis={spec['basis_type']}, score={spec['score_mode']})"
          + (f"  -> {err}" if err else ""))

manifest["code_validation"] = validation_report
save_manifest()

if not all(r["ok"] for r in validation_report):
    raise RuntimeError(
        "02_validate_code_fixes: al menos una combinación falló. "
        "Corrige los bugs señalados antes de continuar con 03+."
    )
print("All combinations validated.")

In [ ]:
# ============================================================
# 03_build_common_hparam_grid
# ============================================================
HIDDEN_OPTIONS = {
    "64_32":      (64, 32),
    "128_64":     (128, 64),
    "192_96":     (192, 96),
    "256_128":    (256, 128),
    "256_128_64": (256, 128, 64),
}

HPARAM_SEARCH_SPACE = {
    "N_BASIS":       (2, 16),                 # int, uniform
    "HIDDEN_KEY":    list(HIDDEN_OPTIONS.keys()),
    "DROPOUT":       (0.0, 0.3),               # float, uniform
    "LR_P0":         (1e-4, 1e-2),             # float, log
    "LR_P1":         (1e-5, 5e-3),             # float, log
    "LAMBDA_SMOOTH": (1e-5, 0.2),              # float, log
    "LAMBDA_ELAST":  (1e-5, 0.2),              # float, log
    "BATCH_SIZE":    [256, 512, 1024],
}

def _sample_config(rng: random.Random) -> dict:
    def log_uniform(lo, hi):
        return float(np.exp(rng.uniform(np.log(lo), np.log(hi))))
    return {
        "N_BASIS":       rng.randint(*HPARAM_SEARCH_SPACE["N_BASIS"]),
        "HIDDEN_KEY":    rng.choice(HPARAM_SEARCH_SPACE["HIDDEN_KEY"]),
        "DROPOUT":       rng.uniform(*HPARAM_SEARCH_SPACE["DROPOUT"]),
        "LR_P0":         log_uniform(*HPARAM_SEARCH_SPACE["LR_P0"]),
        "LR_P1":         log_uniform(*HPARAM_SEARCH_SPACE["LR_P1"]),
        "LAMBDA_SMOOTH": log_uniform(*HPARAM_SEARCH_SPACE["LAMBDA_SMOOTH"]),
        "LAMBDA_ELAST":  log_uniform(*HPARAM_SEARCH_SPACE["LAMBDA_ELAST"]),
        "BATCH_SIZE":    rng.choice(HPARAM_SEARCH_SPACE["BATCH_SIZE"]),
    }

# ── Fixed configuration (nivel 1: fixed-configuration substitution) ──
# Config única y razonable, IDÉNTICA para las 4 arquitecturas. Si existe
# results/best_trial_params.json (de hparam-search.ipynb) se reutiliza,
# normalizando N_KNOTS -> N_BASIS; si no, se usa un fallback documentado.
_best_trial_path = Path("../results/best_trial_params.json")
if _best_trial_path.exists():
    with open(_best_trial_path) as f:
        _raw = json.load(f)
    FIXED_CONFIG = {**_raw, "N_BASIS": _raw.get("N_BASIS", _raw.get("N_KNOTS"))}
    FIXED_CONFIG.pop("N_KNOTS", None)
    fixed_config_source = str(_best_trial_path)
else:
    FIXED_CONFIG = {
        "N_BASIS": 8, "HIDDEN_KEY": "128_64", "DROPOUT": 0.1,
        "LR_P0": 1e-3, "LR_P1": 1e-4, "LAMBDA_SMOOTH": 1e-3, "LAMBDA_ELAST": 1e-2,
        "BATCH_SIZE": 512,
    }
    fixed_config_source = "fallback_default"

missing = set(HPARAM_SEARCH_SPACE) - set(FIXED_CONFIG)
if missing:
    raise ValueError(f"FIXED_CONFIG is missing keys: {missing}")

# ── Common grid (nivel 2: equal-budget retuning) ──────────────────────
# Mismo sampler (random.Random), misma semilla (SAMPLER_SEED), mismo
# espacio de búsqueda -> las N_COMMON_CONFIGS son idénticas para TP-DOT,
# NC-DOT y TP-ADD (screening), eliminando la variabilidad de que cada
# Optuna study explore regiones distintas.
_grid_rng = random.Random(SAMPLER_SEED)
COMMON_HPARAM_GRID = [_sample_config(_grid_rng) for _ in range(N_COMMON_CONFIGS)]

manifest["hparam_grid"] = {
    "fixed_config": FIXED_CONFIG,
    "fixed_config_source": fixed_config_source,
    "search_space": HPARAM_SEARCH_SPACE,
    "common_grid": COMMON_HPARAM_GRID,
}
save_manifest()

print(f"Fixed config (source={fixed_config_source}): {FIXED_CONFIG}")
print(f"Common grid: {len(COMMON_HPARAM_GRID)} configs, sampler_seed={SAMPLER_SEED}")
for i, cfg in enumerate(COMMON_HPARAM_GRID):
    print(f"  cfg{i}: {cfg}")

In [ ]:
# ============================================================
# 04_fixed_config_substitution — parte 1: helpers de entrenamiento
# (idénticos a hparam-search.ipynb; se reutilizan en 04, 05 y 06)
# ============================================================
def build_fold_frames(train_wide, val_wide, smooth_window: int):
    train_wide = train_wide.copy()
    val_wide   = val_wide.copy()
    for w in [train_wide, val_wide]:
        w["store_code"] = w["store_code"].map(store_map)
        w["week_id"]    = w["week_id"].map(week_map)
    train_wide_s = train_wide.sort_values(["store_code", "week_id"]).copy()
    val_wide_s   = val_wide.sort_values(["store_code", "week_id"]).copy()
    for i in range(n_upcs):
        col = f"log_liters_{i}"
        for df_w in [train_wide_s, val_wide_s]:
            df_w[col] = (
                df_w.groupby("store_code")[col]
                .transform(lambda s: s.rolling(window=smooth_window, min_periods=1).mean())
            )
    return train_wide, val_wide, train_wide_s, val_wide_s

def build_loaders(train_wide, val_wide, train_wide_s, val_wide_s, batch_size: int):
    loader_factory = DataLoaderFactory(num_workers=4, pin_memory=True, persistent_workers=True)
    train_ds_p0 = MultiProductDataset(train_wide_s, n=n_upcs)
    val_ds_p0   = MultiProductDataset(val_wide_s,   n=n_upcs)
    train_ds    = MultiProductDataset(train_wide,   n=n_upcs)
    val_ds      = MultiProductDataset(val_wide,     n=n_upcs)
    train_loader_p0 = loader_factory.create_train_loader(train_ds_p0, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader_p0   = loader_factory.create_eval_loader(val_ds_p0,   batch_size=batch_size, shuffle=False)
    train_loader    = loader_factory.create_train_loader(train_ds,    batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader      = loader_factory.create_eval_loader(val_ds,       batch_size=batch_size, shuffle=False)
    return train_loader_p0, val_loader_p0, train_loader, val_loader

def zero_and_freeze_nonlinear(model):
    ph = model.head.param_head
    for attr in ("head_w", "head_w_cross", "head_cross"):
        if not hasattr(ph, attr):
            continue
        layer = getattr(ph, attr)
        with torch.no_grad():
            layer.weight.zero_()
            if layer.bias is not None:
                layer.bias.zero_()
        layer.weight.requires_grad_(False)
        if layer.bias is not None:
            layer.bias.requires_grad_(False)

def unfreeze_nonlinear(model):
    for attr in ["head_w", "head_w_cross", "head_cross"]:
        head = getattr(model.head.param_head, attr)
        head.weight.requires_grad_(True)
        head.bias.requires_grad_(True)

def init_beta_prior(model, beta_target):
    beta_raw_init = torch.log(torch.exp(torch.tensor(-beta_target, dtype=torch.float32)) - 1.0)
    with torch.no_grad():
        model.head.param_head.head_beta.weight.zero_()
        model.head.param_head.head_beta.bias.fill_(beta_raw_init)

def run_training(model, train_loader, val_loader, loss_fn, optimizer, scheduler,
                  n_epochs, es_patience, ckpt_path, device, neighbor_meta,
                  phase_name="", verbose=False):
    import torch.nn as nn
    best_val_loss, no_improve = float("inf"), 0
    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None

    for epoch in range(n_epochs):
        model.train()
        for batch in train_loader:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y_true, obs_mask = batch["demands"], batch["obs_mask"]
            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                    loss, logs = loss_fn(y_hat, y_true, obs_mask, aux["w"], aux["ddBx"], aux["u"], aux["Bx"], aux["pairs"], E=aux.get("E"), attn_weights=aux.get("attn_weights"))
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                loss, logs = loss_fn(y_hat, y_true, obs_mask, aux["w"], aux["ddBx"], aux["u"], aux["Bx"], aux["pairs"], E=aux.get("E"), attn_weights=aux.get("attn_weights"))
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        model.eval()
        val_loss_sum, val_denom = 0.0, 0.0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                y_true, obs_mask = batch["demands"], batch["obs_mask"]
                y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                _, logs = loss_fn(y_hat, y_true, obs_mask, aux["w"], aux["ddBx"], aux["u"], aux["Bx"], aux["pairs"], E=aux.get("E"), attn_weights=aux.get("attn_weights"))
                denom = obs_mask.sum().item()
                val_loss_sum += logs["loss"].item() * denom
                val_denom += denom

        val_loss = val_loss_sum / max(val_denom, 1.0)
        prev_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)
        if optimizer.param_groups[0]["lr"] < prev_lr:
            no_improve = 0
        if val_loss < best_val_loss:
            best_val_loss, no_improve = val_loss, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1
        if verbose and ((epoch + 1) % 50 == 0 or no_improve == 0):
            print(f"  [{phase_name}] Epoch {epoch+1}  val={val_loss:.4f}")
        if no_improve >= es_patience:
            if verbose:
                print(f"  [{phase_name}] Early stopping in epoch {epoch+1}")
            break
    return best_val_loss

def compute_global_metrics(model, val_loader, device):
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y_true, obs_mask = batch["demands"], batch["obs_mask"]
            y_hat, _, _ = model(batch, return_parts=True, neighbor_meta=neighbor_meta)
            mask = obs_mask.bool()
            all_true.append(y_true[mask].cpu())
            all_pred.append(y_hat[mask].cpu())
    y_true_all, y_pred_all = torch.cat(all_true).float(), torch.cat(all_pred).float()
    err = y_true_all - y_pred_all
    mae, rmse = float(err.abs().mean()), float(torch.sqrt((err ** 2).mean()))
    ss_res, ss_tot = float((err ** 2).sum()), float(((y_true_all - y_true_all.mean()) ** 2).sum())
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan
    return {"mae_val": mae, "rmse_val": rmse, "r2_val": r2}

def compute_elasticity_score(model, val_loader, device, own_min=-5.0, own_max=0.0, cross_min=-1.0, cross_max=1.0):
    # Ex-post diagnostic ONLY — nunca se usa para seleccionar configs/arquitecturas (ver 05).
    model.eval()
    all_own, all_cross = [], []
    off_diag = ~torch.eye(model.n, dtype=torch.bool, device=device).unsqueeze(0)
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            _, eps_hat, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
            obs_mask = batch["obs_mask"].bool()
            E = aux["E"]
            all_own.append(eps_hat[obs_mask].cpu())
            all_cross.append(E[active_cross_mask(E, obs_mask, aux["pairs"])].cpu())
    own = torch.cat(all_own).float()
    cross = torch.cat(all_cross).float() if all_cross and all_cross[0].numel() else torch.tensor([])
    own_in_range = float(((own >= own_min) & (own <= own_max)).float().mean())
    cross_in_range = float(((cross >= cross_min) & (cross <= cross_max)).float().mean()) if cross.numel() else float("nan")
    prior_penalty = min(1.0, abs(float(own.median()) - BETA_EDA) / abs(BETA_EDA))
    own_score = own_in_range * (1.0 - prior_penalty)
    elast_score = 0.7 * own_score + 0.3 * (cross_in_range if cross.numel() else 0.0)
    return {
        "elast_score": elast_score, "own_score": own_score, "own_in_range": own_in_range,
        "own_elasticity_median": float(own.median()),
        "cross_in_range": cross_in_range,
        "cross_elasticity_median": float(cross.median()) if cross.numel() else float("nan"),
    }

def active_cross_mask(E, obs_mask, pairs):
    """True only on observed, selected directed edges (not the diagonal)."""
    n = E.shape[1]
    active = torch.zeros(n, n, dtype=torch.bool, device=E.device)
    if pairs is not None and pairs.numel() > 0:
        active[pairs[0], pairs[1]] = True
    obs = obs_mask.bool()
    return obs.unsqueeze(2) & obs.unsqueeze(1) & active.unsqueeze(0)

In [ ]:
def run_two_phase_fit(model_spec, params, train_fold, val_fold, run_type, run_id,
                       fold_id, seed, extract_artifacts=True):
    """Generalización de build_and_train() (hparam-search.ipynb) a cualquier
    (basis_type, score_mode). run_type/run_id/fold_id/seed sólo etiquetan la
    fila de salida para poder concatenar 04/05/06 en un único esquema."""
    set_all_seeds(seed)
    model_id, basis_type, score_mode = model_spec["model_id"], model_spec["basis_type"], model_spec["score_mode"]

    train_wide, val_wide, train_wide_s, val_wide_s = build_fold_frames(train_fold, val_fold, SMOOTH_WINDOW)
    train_loader_p0, val_loader_p0, train_loader, val_loader = build_loaders(
        train_wide, val_wide, train_wide_s, val_wide_s, batch_size=params["BATCH_SIZE"]
    )

    n_basis, hidden, dropout = params["N_BASIS"], HIDDEN_OPTIONS[params["HIDDEN_KEY"]], params["DROPOUT"]
    act = params.get("ACT", "gelu")
    lr_p0, lr_p1 = params["LR_P0"], params["LR_P1"]
    lambda_smooth, lambda_elast = params["LAMBDA_SMOOTH"], params["LAMBDA_ELAST"]

    tag = f"{run_type}_{model_id}_{run_id}_fold{fold_id}_seed{seed}"
    ckpt_p0, ckpt_p1 = CKPT_DIR / f"{tag}_phase0.pt", CKPT_DIR / f"{tag}_phase1.pt"

    builder = SplineBuilder()
    spline_configs = [
        builder.build_from_data(train_wide[f"log_price_{i}"].values,
                                 n_basis=n_basis, q_min=0.05, q_max=0.95, basis_type=basis_type)
        for i in range(n_upcs)
    ]
    price_splines = build_price_basis(basis_type, spline_configs)
    support_bounds = [
    (float(train_wide[f"log_price_{i}"].min()), float(train_wide[f"log_price_{i}"].max()))
    for i in range(n_upcs)
    ]
    token_builder = ProductTokenBuilder(
        n=n_upcs, n_stores=n_stores, d_store=D_STORE,
        n_brands=n_brands, d_brand=D_BRAND, n_styles=n_styles, d_style=D_STYLE,
    )

    def make_model(enforce_negative_beta, use_cross):
        head = IntegrableDemandHead(
            context_dim=token_builder.d_token, K_splines=price_splines.K, n=n_upcs,
            k_neighbors=K_NEIGHBORS, hidden=hidden, act=act, dropout=dropout,
            use_cross=use_cross, enforce_negative_beta=enforce_negative_beta,
            attention_score_mode=score_mode, same_category_strict=False,
        )
        return ICDN(context_builder=token_builder, price_splines=price_splines, head=head, n=n_upcs).to(device)

    def make_optimizer(model, lr):
        decay, no_decay = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            (no_decay if (("head_w" in name) or ("head_cross" in name)
                          or ("head_beta_cross" in name) or name.endswith("bias")) else decay).append(p)
        return torch.optim.AdamW(
            [{"params": decay, "weight_decay": 1e-5}, {"params": no_decay, "weight_decay": 0.0}], lr=lr,
        )

    # ── Phase 0 ──
    m0 = make_model(enforce_negative_beta=True, use_cross=True)
    zero_and_freeze_nonlinear(m0)
    init_beta_prior(m0, BETA_EDA)
    with torch.no_grad():
        m0.head.param_head.head_beta_cross.weight.zero_()
        m0.head.param_head.head_beta_cross.bias.zero_()
    loss_p0 = ElasticityLoss(huber_delta=1.0, lambda_smooth=lambda_smooth, lambda_elast=lambda_elast, reduction="mean")
    opt_p0 = make_optimizer(m0, lr_p0)
    sch_p0 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_p0, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5)
    run_training(m0, train_loader_p0, val_loader_p0, loss_p0, opt_p0, sch_p0,
                 N_EPOCHS_P0, ES_PATIENCE, ckpt_p0, device, neighbor_meta, f"{tag}-P0")

    # ── Phase 1 ──
    m1 = make_model(enforce_negative_beta=True, use_cross=True)
    m1.load_state_dict(torch.load(ckpt_p0, map_location=device))
    unfreeze_nonlinear(m1)
    loss_p1 = ElasticityLoss(huber_delta=1.0, lambda_smooth=lambda_smooth, lambda_elast=lambda_elast, reduction="mean")
    opt_p1 = make_optimizer(m1, lr_p1)
    sch_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_p1, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5)
    run_training(m1, train_loader, val_loader, loss_p1, opt_p1, sch_p1,
                 N_EPOCHS_P1, ES_PATIENCE, ckpt_p1, device, neighbor_meta, f"{tag}-P1")
    m1.load_state_dict(torch.load(ckpt_p1, map_location=device))
    m1.eval()

    # ── Freeze attention graph (idéntico al patrón de hparam-search.ipynb) ──
    selector = m1.head.neighbor_selector
    def h_iter(loader):
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                yield m1.head.encoder(m1.context_builder(batch))
    global_mean = selector.accumulate_mean_scores(
        h_iter(train_loader), category=neighbor_meta["category"], brand=neighbor_meta["brand"],
        style=neighbor_meta["style"], liters=neighbor_meta["liters"],
    )
    selector.freeze_graph(global_mean, category=neighbor_meta["category"], brand=neighbor_meta["brand"],
                           style=neighbor_meta["style"], liters=neighbor_meta["liters"])
    
    # ── Runtime / memory profiling (candidate-scoring vs frozen-graph paths) ──
    import time
    sample_batch = next(iter(val_loader))
    sample_batch = {k: v.to(device, non_blocking=True) for k, v in sample_batch.items()}
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    _frozen_pairs_bak, _frozen_bonus_bak = selector.frozen_pairs, selector.frozen_edge_bonus
    selector.frozen_pairs, selector.frozen_edge_bonus = None, None  # fuerza el camino denso
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = m1(sample_batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
    if device == "cuda": torch.cuda.synchronize()
    candidate_scoring_time_s = time.perf_counter() - t0
    selector.frozen_pairs, selector.frozen_edge_bonus = _frozen_pairs_bak, _frozen_bonus_bak  # restaura
    
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = m1(sample_batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
    if device == "cuda": torch.cuda.synchronize()
    frozen_graph_time_s = time.perf_counter() - t0
    peak_gpu_memory_mb = (torch.cuda.max_memory_allocated() / 1e6) if device == "cuda" else float("nan")

    pred_metrics  = compute_global_metrics(m1, val_loader, device)
    elast_metrics = compute_elasticity_score(m1, val_loader, device)  # sólo diagnóstico ex post

    result = {
        "run_type": run_type, "run_id": run_id, "model_id": model_id,
        "basis_type": basis_type, "score_mode": score_mode,
        "fold": fold_id, "seed": seed, "n_train": len(train_wide), "n_val": len(val_wide),
        "candidate_scoring_time_s": candidate_scoring_time_s,
        "frozen_graph_time_s": frozen_graph_time_s,
        "peak_gpu_memory_mb": peak_gpu_memory_mb,
        **pred_metrics, **elast_metrics,
    }

    artifacts = None
    if extract_artifacts:
        artifacts = extract_run_artifacts(m1, val_loader, run_type, run_id, model_id,
                                        basis_type, score_mode, fold_id, seed,
                                        support_bounds=support_bounds)

    ckpt_p0.unlink(missing_ok=True)
    ckpt_p1.unlink(missing_ok=True)
    return {"metrics": result, "artifacts": artifacts}

from src.nn.loss.components.curvature_calculator import CurvatureCalculator
_curvature_calc = CurvatureCalculator()

def extract_run_artifacts(model, val_loader, run_type, run_id, model_id, basis_type,
                           score_mode, fold_id, seed, support_bounds):
    """Predicciones, elasticidades, edges de atención Y diagnósticos de base
    (curvatura, fuera-de-soporte) de un run ya entrenado."""
    model.eval()
    selector = model.head.neighbor_selector
    n = model.n
    pred_rows, elast_rows = [], []
    entropy_acc = torch.zeros(n, device=device)
    max_weight_acc = torch.zeros(n, device=device)
    n_batches = 0

    kappa_sq_sum, kappa_n = 0.0, 0
    out_of_support = np.zeros(n, dtype=np.int64)
    total_obs = np.zeros(n, dtype=np.int64)
    max_abs_E_out_own, max_abs_E_out_cross = 0.0, 0.0

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y_true, obs_mask = batch["demands"], batch["obs_mask"]
            y_hat, eps_hat, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
            E = aux["E"]
            B = y_hat.shape[0]

            # ── Curvature RMS (misma fórmula que SmoothnessPenalty: mean(kappa^2)) ──
            kappa = _curvature_calc.run(
                aux["w"], aux["ddBx"], aux["u"], aux["Bx"], aux["pairs"],
                aux["attn_weights"],
            )
            kappa_sq_sum += float((kappa ** 2).sum())
            kappa_n += kappa.numel()

            # ── Fuera del soporte de entrenamiento ──────────────────────────────
            x_np = batch["prices"].cpu().numpy()  # (B, n) log-price
            E_np = E.cpu().numpy()
            for i in range(n):
                lo, hi = support_bounds[i]
                out_mask = (x_np[:, i] < lo) | (x_np[:, i] > hi)
                out_of_support[i] += int(out_mask.sum())
                total_obs[i] += x_np.shape[0]
                if out_mask.any():
                    max_abs_E_out_own = max(max_abs_E_out_own, float(np.abs(E_np[out_mask, i, i]).max()))
                    off = [j for j in range(n) if j != i]
                    if off:
                        max_abs_E_out_cross = max(
                            max_abs_E_out_cross, float(np.abs(E_np[out_mask][:, i, off]).max())
                        )

            mask_np = obs_mask.bool().cpu().numpy()
            y_true_np, y_hat_np = y_true.cpu().numpy(), y_hat.cpu().numpy()
            store_np, week_np = batch["store_code"].cpu().numpy(), batch["week_id"].cpu().numpy()

            # Own-price: una fila por producto i
            for i in range(n):
                keep = mask_np[:, i]
                if keep.any():
                    pred_rows.append(pd.DataFrame({
                        "run_type": run_type, "run_id": run_id, "model_id": model_id,
                        "basis_type": basis_type, "score_mode": score_mode,
                        "fold": fold_id, "seed": seed,
                        "store_code": store_np[keep], "week_id": week_np[keep],
                        "upc_i": i, "y_true": y_true_np[keep, i], "y_hat": y_hat_np[keep, i],
                    }))
                elast_rows.append(pd.DataFrame({
                    "run_type": run_type, "run_id": run_id, "model_id": model_id,
                    "basis_type": basis_type, "score_mode": score_mode,
                    "fold": fold_id, "seed": seed,
                    "store_code": store_np, "week_id": week_np,
                    "upc_i": i, "upc_j": i, "type": "own", "E": E_np[:, i, i],
                }))

            # Cross-price: una pasada sobre el grafo activo (fuera del for i)
            pairs_t = aux["pairs"]
            if pairs_t is not None and pairs_t.numel() > 0:
                i_idx = pairs_t[0].tolist()
                j_idx = pairs_t[1].tolist()
                for ii, jj in zip(i_idx, j_idx):
                    elast_rows.append(pd.DataFrame({
                        "run_type": run_type, "run_id": run_id, "model_id": model_id,
                        "basis_type": basis_type, "score_mode": score_mode,
                        "fold": fold_id, "seed": seed,
                        "store_code": store_np, "week_id": week_np,
                        "upc_i": ii, "upc_j": jj, "type": "cross", "E": E_np[:, ii, jj],
                    }))

            if selector.frozen_pairs is not None:
                pairs = selector.frozen_pairs
                k_eff = pairs.shape[1] // n
                _, edge_weights = selector.run(
                    h=model.head.encoder(model.context_builder(batch)),
                    category=neighbor_meta["category"], brand=neighbor_meta["brand"],
                    style=neighbor_meta["style"], liters=neighbor_meta["liters"],
                )
                w = edge_weights.view(B, n, k_eff).clamp_min(1e-12)
                entropy_acc    += -(w * w.log()).sum(-1).mean(0)
                max_weight_acc += w.max(-1).values.mean(0)
                n_batches += 1

    pred_df  = pd.concat(pred_rows,  ignore_index=True) if pred_rows  else pd.DataFrame()
    elast_df = pd.concat(elast_rows, ignore_index=True) if elast_rows else pd.DataFrame()

    edges_df = pd.DataFrame()
    if selector.frozen_pairs is not None and n_batches > 0:
        pairs_np = selector.frozen_pairs.cpu().numpy()
        mean_entropy = (entropy_acc / n_batches).cpu().numpy()
        mean_max_w   = (max_weight_acc / n_batches).cpu().numpy()
        edges_df = pd.DataFrame({
            "run_type": run_type, "run_id": run_id, "model_id": model_id,
            "basis_type": basis_type, "score_mode": score_mode,
            "fold": fold_id, "seed": seed,
            "i_idx": pairs_np[0], "j_idx": pairs_np[1],
            "mean_entropy_i": mean_entropy[pairs_np[0]],
            "mean_max_weight_i": mean_max_w[pairs_np[0]],
        })

    basis_diag_df = pd.DataFrame([{
        "run_type": run_type, "run_id": run_id, "model_id": model_id,
        "basis_type": basis_type, "score_mode": score_mode, "fold": fold_id, "seed": seed,
        "curvature_rms": float(np.sqrt(kappa_sq_sum / max(kappa_n, 1))),
        "pct_out_of_support": float(out_of_support.sum() / max(total_obs.sum(), 1)),
        "max_abs_own_E_out_of_support": max_abs_E_out_own,
        "max_abs_cross_E_out_of_support": max_abs_E_out_cross,
    }])

    return {"predictions": pred_df, "elasticities": elast_df, "edges": edges_df, "basis_diag": basis_diag_df}

In [ ]:
# ============================================================
# 04_fixed_config_substitution — parte 2
# ============================================================
# Nivel 1: FIXED_CONFIG idéntica para las 4 arquitecturas (incluye NC-ADD,
# diagnóstico factorial), sobre el fold de referencia (outer fold 0) y las
# seeds pareadas CONFIRMATORY_SEEDS.
fixed_config_rows = []
fixed_config_artifacts = {"predictions": [], "elasticities": [], "edges": [], "basis_diag": []}
ref_train_fold, ref_val_fold = outer_fold_splits[0]

for spec in MODEL_MATRIX:
    for seed in CONFIRMATORY_SEEDS:
        print(f"[04] {spec['model_id']} seed={seed}")
        out = run_two_phase_fit(
            model_spec=spec, params=FIXED_CONFIG,
            train_fold=ref_train_fold, val_fold=ref_val_fold,
            run_type="fixed_config", run_id="ref_fold0", fold_id=0, seed=seed,
        )
        fixed_config_rows.append(out["metrics"])
        for key in fixed_config_artifacts:
            fixed_config_artifacts[key].append(out["artifacts"][key])
        print(f"    R2={out['metrics']['r2_val']:.4f} MAE={out['metrics']['mae_val']:.4f}")

fixed_config_df = pd.DataFrame(fixed_config_rows)
fixed_config_df.to_parquet(FIXED_CONFIG_PATH, index=False)
manifest["fixed_config_substitution"] = {"n_runs": len(fixed_config_df), "ref_fold": 0, "seeds": CONFIRMATORY_SEEDS}
save_manifest()
print(f"Saved {len(fixed_config_df)} rows -> {FIXED_CONFIG_PATH}")
fixed_config_df

In [ ]:
# ============================================================
# 05_equal_budget_search
# ============================================================
# Screening: SCREENING_MODELS (3, sin NC-ADD) x COMMON_HPARAM_GRID x
# inner_fold_splits x SCREENING_SEEDS. Selección EXCLUSIVAMENTE predictiva
# (robust R2) — nunca elast_score (ver justificación en el diseño).
search_rows = []
screening_specs = [s for s in MODEL_MATRIX if s["model_id"] in SCREENING_MODELS]

for spec in screening_specs:
    for cfg_id, cfg in enumerate(COMMON_HPARAM_GRID):
        for fold_id, (train_fold, val_fold) in enumerate(inner_fold_splits):
            for seed in SCREENING_SEEDS:
                print(f"[05] {spec['model_id']} cfg{cfg_id} fold{fold_id} seed={seed}")
                out = run_two_phase_fit(
                    model_spec=spec, params=cfg,
                    train_fold=train_fold, val_fold=val_fold,
                    run_type="search", run_id=f"cfg{cfg_id}", fold_id=fold_id, seed=seed,
                    extract_artifacts=False,   # screening: sólo métricas, sin elasticidades/edges
                )
                search_rows.append({**out["metrics"], "cfg_id": cfg_id})

search_df = pd.DataFrame(search_rows)
search_df.to_parquet(SEARCH_RUNS_PATH, index=False)
print(f"Saved {len(search_df)} rows -> {SEARCH_RUNS_PATH}")

# ── Selección predictiva-only ──────────────────────────────────────────
agg = (
    search_df.groupby(["model_id", "cfg_id"])
    .agg(mean_r2=("r2_val", "mean"), std_r2=("r2_val", "std"),
         mean_mae=("mae_val", "mean"), std_mae=("mae_val", "std"))
    .reset_index()
)
agg["std_r2"] = agg["std_r2"].fillna(0.0)
agg["robust_r2"] = agg["mean_r2"] - 0.25 * agg["std_r2"]

best_config_per_model = {}
for model_id, group in agg.groupby("model_id"):
    best_row = group.sort_values("robust_r2", ascending=False).iloc[0]
    best_cfg_id = int(best_row["cfg_id"])
    best_config_per_model[model_id] = {
        "cfg_id": best_cfg_id, "config": COMMON_HPARAM_GRID[best_cfg_id],
        "robust_r2": float(best_row["robust_r2"]), "mean_mae": float(best_row["mean_mae"]),
    }
    print(f"[05] Best for {model_id}: cfg{best_cfg_id} robust_r2={best_row['robust_r2']:.4f} mean_mae={best_row['mean_mae']:.4f}")

manifest["equal_budget_search"] = {
    "n_runs": len(search_df),
    "selection_criterion": "robust_r2 = mean_r2 - 0.25*std_r2 (predictivo, sin elast_score)",
    "best_config_per_model": best_config_per_model,
}
save_manifest()

In [ ]:
# ============================================================
# 06_outer_fold_confirmatory_runs
# ============================================================
# Confirmatory: CONFIRMATORY_MODELS (3) con SU propia mejor config (de 05),
# sobre outer_fold_splits x CONFIRMATORY_SEEDS pareadas.
outer_rows = []
outer_artifacts    = {"predictions": [], "elasticities": [], "edges": [], "basis_diag": []}
confirmatory_specs = [s for s in MODEL_MATRIX if s["model_id"] in CONFIRMATORY_MODELS]

for spec in confirmatory_specs:
    cfg = best_config_per_model[spec["model_id"]]["config"]
    for fold_id, (train_fold, val_fold) in enumerate(outer_fold_splits):
        for seed in CONFIRMATORY_SEEDS:
            print(f"[06] {spec['model_id']} fold{fold_id} seed={seed}")
            out = run_two_phase_fit(
                model_spec=spec, params=cfg,
                train_fold=train_fold, val_fold=val_fold,
                run_type="confirmatory", run_id="outer", fold_id=fold_id, seed=seed,
            )
            outer_rows.append(out["metrics"])
            for key in outer_artifacts:
                outer_artifacts[key].append(out["artifacts"][key])
            print(f"    R2={out['metrics']['r2_val']:.4f} MAE={out['metrics']['mae_val']:.4f}")

outer_metrics_df = pd.DataFrame(outer_rows)
outer_metrics_df.to_parquet(OUTER_METRICS_PATH, index=False)
manifest["outer_fold_confirmatory"] = {"n_runs": len(outer_metrics_df), "n_outer_folds": N_OUTER_FOLDS, "seeds": CONFIRMATORY_SEEDS}
save_manifest()
print(f"Saved {len(outer_metrics_df)} rows -> {OUTER_METRICS_PATH}")
outer_metrics_df

In [ ]:
# ============================================================
# 07_extract_predictions_elasticities_edges
# ============================================================
# Concatena los artefactos ya extraídos en 04 (fixed_config) y 06
# (confirmatory). 05 (screening) no extrae elasticidades/edges — sólo
# se usó para seleccionar config por criterio predictivo.
def _concat_artifacts(*artifact_dicts, key):
    frames = [f for d in artifact_dicts for f in d[key] if not f.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

all_predictions  = _concat_artifacts(fixed_config_artifacts, outer_artifacts, key="predictions")
all_elasticities = _concat_artifacts(fixed_config_artifacts, outer_artifacts, key="elasticities")
all_edges        = _concat_artifacts(fixed_config_artifacts, outer_artifacts, key="edges")
all_basis_diag = _concat_artifacts(fixed_config_artifacts, outer_artifacts, key="basis_diag")

all_elasticities.to_parquet(ELASTICITIES_PATH, index=False)
all_edges.to_parquet(ATTENTION_EDGES_PATH, index=False)

print(f"predictions:  {len(all_predictions):,} rows (en memoria, alimenta 08; no está en la lista de outputs)")
print(f"elasticities: {len(all_elasticities):,} rows -> {ELASTICITIES_PATH}")
print(f"attention_edges: {len(all_edges):,} rows -> {ATTENTION_EDGES_PATH}")

manifest["extraction"] = {
    "n_prediction_rows": len(all_predictions),
    "n_elasticity_rows": len(all_elasticities),
    "n_edge_rows": len(all_edges),
}
save_manifest()

In [ ]:
# ============================================================
# 08_paired_statistics
# ============================================================
N_BOOTSTRAP = 20 if SMOKE_TEST else 200
REFERENCE_MODEL = "TP-DOT"   # baseline: truncated-power + scaled dot-product (arquitectura actual)
BLOCK_SIZE_WEEKS = 4

def paired_bootstrap_mae(all_predictions, model_ids, n_boot, block_size, base_seed=0):
    """Bootstrap por bloques de semanas, PAREADO entre arquitecturas: para cada
    (draw b, fold f) se usa la misma rng -> las mismas semanas resampleadas en
    todas las arquitecturas, así ΔMAE refleja el mismo resample, no ruido extra."""
    conf = all_predictions[all_predictions["run_type"] == "confirmatory"]
    folds = sorted(conf["fold"].unique())
    boot_mae = {m: [] for m in model_ids}
    for b in range(n_boot):
        for m in model_ids:
            df_m = conf[conf["model_id"] == m]
            pieces = []
            for f in folds:
                df_mf = df_m[df_m["fold"] == f]
                if df_mf.empty:
                    continue
                weeks = df_mf["week_id"].unique()
                rng = np.random.default_rng(base_seed * 100_000 + b * 1_000 + f)
                sampler = BlockBootstrapSampler(week_col="week_id", block_size=block_size, rng=rng)
                pieces.append(sampler.sample(df_mf, train_weeks=weeks))
            resampled = pd.concat(pieces, ignore_index=True) if pieces else df_m
            boot_mae[m].append(float((resampled["y_true"] - resampled["y_hat"]).abs().mean()))
    return {m: np.array(v) for m, v in boot_mae.items()}

boot_mae = paired_bootstrap_mae(all_predictions, CONFIRMATORY_MODELS, N_BOOTSTRAP, BLOCK_SIZE_WEEKS)
ref_draws = boot_mae[REFERENCE_MODEL]

paired_stats_rows = []
for m in CONFIRMATORY_MODELS:
    sub = outer_metrics_df[outer_metrics_df.model_id == m]
    delta_draws = boot_mae[m] - ref_draws
    paired_stats_rows.append({
        "model_id": m,
        "mae": float(sub["mae_val"].mean()),
        "mae_ci_low": float(np.percentile(boot_mae[m], 2.5)),
        "mae_ci_high": float(np.percentile(boot_mae[m], 97.5)),
        "delta_mae_vs_ref": float(delta_draws.mean()),
        "delta_mae_ci_low": float(np.percentile(delta_draws, 2.5)),
        "delta_mae_ci_high": float(np.percentile(delta_draws, 97.5)),
        "r2": float(sub["r2_val"].mean()),
        "inter_seed_sd": float(sub.groupby("seed")["mae_val"].mean().std()),
        "inter_fold_sd": float(sub.groupby("fold")["mae_val"].mean().std()),
    })
paired_stats_df = pd.DataFrame(paired_stats_rows)
print(f"Paired bootstrap (n_boot={N_BOOTSTRAP}, reference={REFERENCE_MODEL}):")
paired_stats_df

In [ ]:
# ── Diagnósticos comunes (suplemento) ──────────────────────────────────
def common_diagnostics(elasticities_df, run_type="confirmatory"):
    df = elasticities_df[elasticities_df["run_type"] == run_type]
    rows = []
    for (m, t), g in df.groupby(["model_id", "type"]):
        lo, hi = (-5.0, 0.0) if t == "own" else (-1.0, 1.0)
        rows.append({
            "model_id": m, "type": t,
            "pct_negative": float((g["E"] < 0).mean()) if t == "own" else np.nan,
            "pct_in_range": float(((g["E"] >= lo) & (g["E"] <= hi)).mean()),
            "median": float(g["E"].median()),
            "iqr": float(g["E"].quantile(0.75) - g["E"].quantile(0.25)),
        })
    return pd.DataFrame(rows)

common_diag_df = common_diagnostics(all_elasticities)
common_diag_df

In [ ]:
# ============================================================
# 09_basis_boundary_diagnostics
# ============================================================
# Requiere el parche (curvature_rms, pct_out_of_support, etc. en all_basis_diag).
# Compara TP-* (truncated_cubic) vs NC-* (natural_cubic): la hipótesis de la
# Propuesta 1 es que NC-* debería mostrar curvature_rms y max|E| fuera de
# soporte más bajos (extrapolación lineal en vez de crecimiento cúbico).
basis_diag_summary = (
    all_basis_diag[all_basis_diag["run_type"] == "confirmatory"]
    .groupby("model_id")
    .agg(
        curvature_rms_mean=("curvature_rms", "mean"),
        curvature_rms_std=("curvature_rms", "std"),
        pct_out_of_support_mean=("pct_out_of_support", "mean"),
        max_abs_own_E_out=("max_abs_own_E_out_of_support", "max"),
        max_abs_cross_E_out=("max_abs_cross_E_out_of_support", "max"),
    )
    .reset_index()
)
print("Curvatura y comportamiento fuera del soporte de entrenamiento (confirmatory):")
basis_diag_summary

In [ ]:
def near_bound_fraction(elasticities_df, run_type="confirmatory", tol=0.25):
    df = elasticities_df[elasticities_df["run_type"] == run_type]
    rows = []
    for (m, t), g in df.groupby(["model_id", "type"]):
        lo, hi = (-5.0, 0.0) if t == "own" else (-1.0, 1.0)
        near = (np.abs(g["E"] - lo) < tol) | (np.abs(g["E"] - hi) < tol)
        rows.append({"model_id": m, "type": t, "pct_near_bound": float(near.mean())})
    return pd.DataFrame(rows)

near_bound_df = near_bound_fraction(all_elasticities)
near_bound_df

In [ ]:
# ============================================================
# 10_attention_graph_diagnostics
# ============================================================
def edge_set_by_product(edges_df, model_id, fold=None, seed=None, run_type="confirmatory"):
    df = edges_df[(edges_df.model_id == model_id) & (edges_df.run_type == run_type)]
    if fold is not None:
        df = df[df.fold == fold]
    if seed is not None:
        df = df[df.seed == seed]
    return df.groupby("i_idx")["j_idx"].apply(lambda s: frozenset(s.tolist())).to_dict()

def jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)

def mean_jaccard_across(edges_df, model_id, group_col, fixed_col, fixed_val):
    """Jaccard medio (por producto focal i) entre todos los pares de valores
    de group_col (seeds o folds), manteniendo fixed_col fijo."""
    values = sorted(edges_df.loc[
        (edges_df.model_id == model_id) & (edges_df[fixed_col] == fixed_val), group_col
    ].unique())
    if len(values) < 2:
        return np.nan
    scores = []
    for a in range(len(values)):
        for b in range(a + 1, len(values)):
            sets_a = edge_set_by_product(edges_df, model_id, **{fixed_col: fixed_val, group_col: values[a]})
            sets_b = edge_set_by_product(edges_df, model_id, **{fixed_col: fixed_val, group_col: values[b]})
            for i in set(sets_a) | set(sets_b):
                scores.append(jaccard(sets_a.get(i, frozenset()), sets_b.get(i, frozenset())))
    return float(np.mean(scores)) if scores else np.nan

attention_diag_rows = []
for model_id in CONFIRMATORY_MODELS:
    for fold in sorted(all_edges.loc[all_edges.model_id == model_id, "fold"].unique()):
        jac = mean_jaccard_across(all_edges, model_id, group_col="seed", fixed_col="fold", fixed_val=fold)
        attention_diag_rows.append({"model_id": model_id, "axis": "seeds", "fixed_value": fold, "mean_jaccard": jac})
    for seed in sorted(all_edges.loc[all_edges.model_id == model_id, "seed"].unique()):
        jac = mean_jaccard_across(all_edges, model_id, group_col="fold", fixed_col="seed", fixed_val=seed)
        attention_diag_rows.append({"model_id": model_id, "axis": "folds", "fixed_value": seed, "mean_jaccard": jac})

attention_jaccard_df = pd.DataFrame(attention_diag_rows)
attention_entropy_df = (
    all_edges[all_edges.run_type == "confirmatory"]
    .groupby("model_id")
    .agg(mean_entropy=("mean_entropy_i", "mean"), mean_max_weight=("mean_max_weight_i", "mean"))
    .reset_index()
)

print("Jaccard top-k medio por eje (seeds / folds):")
print(attention_jaccard_df.groupby(["model_id", "axis"])["mean_jaccard"].mean())
print("\nEntropía media / máxima ponderación por producto (confirmatory):")
attention_entropy_df

In [ ]:
# ============================================================
# 11_runtime_memory
# ============================================================
outer_metrics_df["speedup_frozen_vs_candidate"] = (
    outer_metrics_df["candidate_scoring_time_s"] / outer_metrics_df["frozen_graph_time_s"]
)
runtime_summary = (
    outer_metrics_df.groupby("model_id")
    .agg(
        candidate_scoring_time_s_mean=("candidate_scoring_time_s", "mean"),
        frozen_graph_time_s_mean=("frozen_graph_time_s", "mean"),
        speedup_frozen_vs_candidate_mean=("speedup_frozen_vs_candidate", "mean"),
        peak_gpu_memory_mb_mean=("peak_gpu_memory_mb", "mean"),
        peak_gpu_memory_mb_max=("peak_gpu_memory_mb", "max"),
    )
    .reset_index()
)
manifest["runtime_memory"] = runtime_summary.to_dict(orient="records")
save_manifest()
print("Runtime / memoria (confirmatory; 1 batch de val_loader por run):")
runtime_summary

In [ ]:
# ============================================================
# 12_export_tables_and_figures
# ============================================================
def p99_abs_E(elasticities_df, run_type="confirmatory"):
    df = elasticities_df[elasticities_df["run_type"] == run_type]
    rows = []
    for (m, t), g in df.groupby(["model_id", "type"]):
        rows.append({"model_id": m, "type": t, "p99_abs_E": float(g["E"].abs().quantile(0.99))})
    return pd.DataFrame(rows).pivot(index="model_id", columns="type", values="p99_abs_E").reset_index()

p99_df = p99_abs_E(all_elasticities).rename(columns={"cross": "cross_p99_abs_E", "own": "own_p99_abs_E"})

summary_table = (
    paired_stats_df
    .merge(p99_df, on="model_id", how="left")
    .merge(basis_diag_summary[["model_id", "curvature_rms_mean"]], on="model_id", how="left")
    .merge(runtime_summary[["model_id", "frozen_graph_time_s_mean", "peak_gpu_memory_mb_mean"]], on="model_id", how="left")
)

summary_table["DeltaMAE_95CI"] = summary_table.apply(
    lambda r: f"{r['delta_mae_vs_ref']:+.4f} [{r['delta_mae_ci_low']:+.4f}, {r['delta_mae_ci_high']:+.4f}]", axis=1
)
summary_table = summary_table.rename(columns={
    "model_id": "Modelo", "mae": "MAE", "r2": "R2",
    "curvature_rms_mean": "Curvature_RMS",
    "inter_seed_sd": "Inter_seed_SD", "inter_fold_sd": "Inter_fold_SD",
    "frozen_graph_time_s_mean": "Tiempo_s", "peak_gpu_memory_mb_mean": "Memoria_MB",
})

final_cols = ["Modelo", "MAE", "DeltaMAE_95CI", "R2", "own_p99_abs_E", "cross_p99_abs_E",
              "Curvature_RMS", "Inter_seed_SD", "Inter_fold_SD", "Tiempo_s", "Memoria_MB"]
summary_table = summary_table[final_cols]
summary_table.to_csv(SUMMARY_TABLE_PATH, index=False)

manifest["outputs"] = {
    "run_manifest": str(MANIFEST_PATH), "fixed_config_runs": str(FIXED_CONFIG_PATH),
    "search_runs": str(SEARCH_RUNS_PATH), "outer_metrics": str(OUTER_METRICS_PATH),
    "elasticities": str(ELASTICITIES_PATH), "attention_edges": str(ATTENTION_EDGES_PATH),
    "summary_table": str(SUMMARY_TABLE_PATH),
}
manifest["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
save_manifest()

print(f"Saved -> {SUMMARY_TABLE_PATH}")
summary_table

In [ ]:
# ============================================================
# 13_visualization
# ============================================================
import matplotlib.pyplot as plt

FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. MAE con IC 95% por arquitectura ─────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
y_pos = np.arange(len(paired_stats_df))
ax.errorbar(
    paired_stats_df["mae"], y_pos,
    xerr=[paired_stats_df["mae"] - paired_stats_df["mae_ci_low"],
          paired_stats_df["mae_ci_high"] - paired_stats_df["mae"]],
    fmt="o", capsize=4,
)
ax.set_yticks(y_pos); ax.set_yticklabels(paired_stats_df["model_id"])
ax.set_xlabel("MAE (val, confirmatory)"); ax.set_title("MAE con IC bootstrap 95%")
ax.invert_yaxis()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "01_mae_ci.png", dpi=150); plt.show()

# ── 2. ΔMAE vs TP-DOT (forest plot) ────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 3))
ref_rows = paired_stats_df[paired_stats_df.model_id != REFERENCE_MODEL]
y_pos = np.arange(len(ref_rows))
ax.errorbar(
    ref_rows["delta_mae_vs_ref"], y_pos,
    xerr=[ref_rows["delta_mae_vs_ref"] - ref_rows["delta_mae_ci_low"],
          ref_rows["delta_mae_ci_high"] - ref_rows["delta_mae_vs_ref"]],
    fmt="s", color="tab:orange", capsize=4,
)
ax.axvline(0, color="gray", linestyle="--", linewidth=1)
ax.set_yticks(y_pos); ax.set_yticklabels(ref_rows["model_id"])
ax.set_xlabel(f"ΔMAE vs {REFERENCE_MODEL}"); ax.set_title("Efecto de la sustitución arquitectónica")
ax.invert_yaxis()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "02_delta_mae_forest.png", dpi=150); plt.show()

# ── 3. Estabilidad temporal: R2 por outer fold ─────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
for m in CONFIRMATORY_MODELS:
    sub = outer_metrics_df[outer_metrics_df.model_id == m].groupby("fold")["r2_val"].mean()
    ax.plot(sub.index, sub.values, marker="o", label=m)
ax.set_xlabel("Outer fold (temporal, expanding)"); ax.set_ylabel("R2 (val)")
ax.set_title("Estabilidad temporal de R2"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "03_r2_by_fold.png", dpi=150); plt.show()

# ── 4. Distribución de elasticidades own/cross ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, t, bounds in zip(axes, ["own", "cross"], [(-5, 0), (-1, 1)]):
    data = [
        all_elasticities.loc[
            (all_elasticities.run_type == "confirmatory") &
            (all_elasticities.model_id == m) & (all_elasticities.type == t), "E"
        ].values
        for m in CONFIRMATORY_MODELS
    ]
    ax.boxplot(data, labels=CONFIRMATORY_MODELS, showfliers=False)
    ax.axhline(bounds[0], color="red", linestyle="--", linewidth=1)
    ax.axhline(bounds[1], color="red", linestyle="--", linewidth=1)
    ax.set_title(f"Elasticidad {t}"); ax.set_ylabel("E")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "04_elasticity_distributions.png", dpi=150); plt.show()

# ── 5. Curvatura vs % fuera de soporte (hipótesis Propuesta 1) ─────────
fig, ax = plt.subplots(figsize=(6, 4))
diag = all_basis_diag[all_basis_diag.run_type == "confirmatory"]
for m in CONFIRMATORY_MODELS:
    sub = diag[diag.model_id == m]
    ax.scatter(sub["pct_out_of_support"], sub["curvature_rms"], label=m, s=60)
ax.set_xlabel("% observaciones fuera del soporte de entrenamiento")
ax.set_ylabel("Curvature RMS")
ax.set_title("Truncated-power vs natural-cubic: curvatura fuera de soporte")
ax.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "05_curvature_vs_support.png", dpi=150); plt.show()

# ── 6. Jaccard top-k medio (seeds/folds) por arquitectura ───────────────
fig, ax = plt.subplots(figsize=(6, 4))
jac_summary = attention_jaccard_df.groupby(["model_id", "axis"])["mean_jaccard"].mean().unstack()
jac_summary.plot(kind="bar", ax=ax)
ax.set_ylabel("Jaccard medio"); ax.set_title("Estabilidad del grafo de atención")
ax.set_ylim(0, 1)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "06_attention_jaccard.png", dpi=150); plt.show()

# ── 7. Tiempo (frozen-graph) y memoria pico por arquitectura ────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(runtime_summary["model_id"], runtime_summary["frozen_graph_time_s_mean"])
axes[0].set_ylabel("Tiempo (s) — frozen graph"); axes[0].set_title("Latencia de inferencia")
axes[1].bar(runtime_summary["model_id"], runtime_summary["peak_gpu_memory_mb_mean"], color="tab:green")
axes[1].set_ylabel("Memoria GPU pico (MB)"); axes[1].set_title("Memoria")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "07_runtime_memory.png", dpi=150); plt.show()

print(f"Figuras guardadas en {FIGURES_DIR}")